# Gold Fact Table Route

In [6]:
from pyspark.sql.functions import col, count, sum, when, round, length

# Load sources
df_flights = spark.table("EAA_LakeHouse.silver.opdi_flights")
df_atfm = spark.table("EAA_LakeHouse.silver.atfm_delays")

# Filter valid routes only — both ADEP and ADES must be 4-char ICAO codes
df_routes = df_flights.filter(
    (length(col("adep_icao")) == 4) &
    (length(col("ades_icao")) == 4)
)

print(f"Total flights before filter : {df_flights.count()}")
print(f"Valid route flights          : {df_routes.count()}")

StatementMeta(, 36209b44-7261-441b-b988-dfedbdccc4e9, 8, Finished, Available, Finished, False)

Total flights before filter : 31733131
Valid route flights          : 14095185


In [7]:
from pyspark.sql.functions import count, sum, when, round

df_agg = (
    df_routes
    .groupBy("adep_icao", "ades_icao", "year_month")
    .agg(
        count("flight_id").alias("total_flights"),
        sum("lcc_flag").alias("lcc_flights"),
        sum("network_flag").alias("network_flights"),
        sum("regional_flag").alias("regional_flights"),
        sum("other_flag").alias("other_flights"),
        sum(when(
    (col("lcc_flag") + col("network_flag") + col("regional_flag") + col("other_flag")) == 0,1).otherwise(0)).alias("unknown_flights")
    )
    .withColumn(
        "lcc_share_pct",
        round(col("lcc_flights") / col("total_flights") * 100, 2)
    )
)

print(f"Distinct routes × month : {df_agg.count()}")
display(df_agg.orderBy(col("total_flights").desc()).limit(10))

StatementMeta(, 36209b44-7261-441b-b988-dfedbdccc4e9, 9, Finished, Available, Finished, False)

Distinct routes × month : 985609


SynapseWidget(Synapse.DataFrame, c497d1d8-ed22-46c1-aabb-0f9f2adfd274)

In [8]:
from pyspark.sql.functions import avg, sum as spark_sum

# Aggregate ATFM to airport × month grain
df_atfm_month = (
    df_atfm
    .groupBy("APT_ICAO", "year_month")
    .agg(
        spark_sum("TOTAL_ATFM_DELAY_MINUTES").alias("total_delay_min"),
        spark_sum("TOTAL_ARRIVALS").alias("total_arrivals")
    )
    .withColumn(
        "avg_atfm_delay_per_arrival",
        round(col("total_delay_min") / col("total_arrivals"), 2)
    )
)

# Join on ADEP
df_with_adep = (
    df_agg
    .join(
        df_atfm_month.select(
            col("APT_ICAO").alias("adep_icao"),
            col("year_month"),
            col("avg_atfm_delay_per_arrival").alias("avg_atfm_delay_adep")
        ),
        on=["adep_icao", "year_month"],
        how="left"
    )
)

# Join on ADES
df_fact = (
    df_with_adep
    .join(
        df_atfm_month.select(
            col("APT_ICAO").alias("ades_icao"),
            col("year_month"),
            col("avg_atfm_delay_per_arrival").alias("avg_atfm_delay_ades")
        ),
        on=["ades_icao", "year_month"],
        how="left"
    )
)

print(f"Row count after joins : {df_fact.count()}")
display(df_fact.orderBy(col("total_flights").desc()).limit(10))

StatementMeta(, 36209b44-7261-441b-b988-dfedbdccc4e9, 10, Finished, Available, Finished, False)

Row count after joins : 985609


SynapseWidget(Synapse.DataFrame, eabeee62-5e36-49d4-8296-b5b13629e3ed)

In [9]:
df_fact.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_route_month")

StatementMeta(, 36209b44-7261-441b-b988-dfedbdccc4e9, 11, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import col, sum, count, min, max

df = spark.table("gold.fact_route_month")
errors = []

# CRITICAL — no null grain keys
for c in ["adep_icao", "ades_icao", "year_month"]:
    n = df.filter(col(c).isNull()).count()
    if n > 0:
        errors.append(f"FAIL: {n} null {c}")

# CRITICAL — no duplicate grain
dupes = df.groupBy("adep_icao", "ades_icao", "year_month").count().filter(col("count") > 1).count()
if dupes > 0:
    errors.append(f"FAIL: {dupes} duplicate grain rows")

# CRITICAL — carrier flags sum must equal total_flights
flag_sum = col("lcc_flights") + col("network_flights") + col("regional_flights") + col("other_flights") + col("unknown_flights")
inconsistent = df.filter(flag_sum != col("total_flights")).count()
if inconsistent > 0:
    errors.append(f"FAIL: {inconsistent} rows where carrier flags != total_flights")

# INFO — year_month range
display(df.agg(min("year_month").alias("min_ym"), max("year_month").alias("max_ym")))

if errors:
    raise ValueError("\n".join(errors))
else:
    print(f"All checks passed — {df.count()} rows")

StatementMeta(, 36209b44-7261-441b-b988-dfedbdccc4e9, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe0ed09b-00a9-4a86-973b-2c8d7e3374e0)

All checks passed — 985609 rows
